# 🧬 Workshop 2: Core NLP & Retrieval-Augmented Generation (RAG)

Welcome to **Workshop 2: Core NLP & RAG**. This notebook covers how raw text is preprocessed into sub-word tokens, mapped to continuous vector representations (embeddings), searched using semantic similarity, and fed into an LLM context as grounded data (RAG).

## 🗺️ Learning Agenda

1. **Tokenization:** Splitting text into numerical token indexes.
2. **Embeddings:** High-dimensional semantics and cosine similarity computations.
3. **Attention Mechanism:** Queries, Keys, Values (Q, K, V) context weights.
4. **Chunking & Vector Databases:** Splitting text documents and approximate nearest neighbor search.
5. **In-Memory RAG Pipeline:** Connecting document retrieval to LLM generation prompts.

---

## 1. Tokenization

Large Language Models do not read text directly. Instead, they process sequences of **Tokens** (sub-word representations). Let's write a simple Word-Level and Character-Level Tokenizer from scratch to understand vocabulary index mapping.

In [ ]:
class SimpleTokenizer:
    def __init__(self):
        self.vocab = {"<PAD>": 0, "<UNK>": 1}
        self.idx_to_token = {0: "<PAD>", 1: "<UNK>"}
        
    def build_vocab(self, corpus):
        # Simple word splitting
        words = sorted(list(set(corpus.lower().split())))
        for i, word in enumerate(words):
            idx = i + 2
            self.vocab[word] = idx
            self.idx_to_token[idx] = word
            
    def encode(self, text):
        tokens = []
        for word in text.lower().split():
            tokens.append(self.vocab.get(word, 1)) # Default to <UNK>
        return tokens
        
    def decode(self, ids):
        return " ".join([self.idx_to_token.get(idx, "<UNK>") for idx in ids])

# Sample Corpus
corpus = "attention is all you need to build intelligent machines"
tokenizer = SimpleTokenizer()
tokenizer.build_vocab(corpus)

input_text = "attention is build need"
encoded_ids = tokenizer.encode(input_text)
decoded_text = tokenizer.decode(encoded_ids)

print(f"Vocabulary: {tokenizer.vocab}")
print(f"Input Text: '{input_text}'")
print(f"Encoded IDs: {encoded_ids}")
print(f"Decoded Text: '{decoded_text}'")

---

## 2. Embeddings & Cosine Similarity

An **embedding** maps semantic meaning to a high-dimensional vector. To measure how similar two meanings are, we compute **Cosine Similarity** between their vectors:
$$\text{Cosine Similarity}(u, v) = \frac{u \cdot v}{\|u\| \|v\|} = \frac{\sum u_i v_i}{\sqrt{\sum u_i^2} \sqrt{\sum v_i^2}}$$

In [ ]:
import numpy as np

def cosine_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

# 3-dimensional mock embeddings for words
# Dimension 1: Royalty, Dimension 2: Femininity, Dimension 3: Food-nature
embeddings = {
    "king":   np.array([0.9, 0.1, 0.0]),
    "queen":  np.array([0.9, 0.9, 0.0]),
    "apple":  np.array([0.0, 0.1, 0.95]),
    "orange": np.array([0.0, 0.1, 0.90])
}

sim_king_queen = cosine_similarity(embeddings["king"], embeddings["queen"])
sim_king_apple = cosine_similarity(embeddings["king"], embeddings["apple"])
sim_apple_orange = cosine_similarity(embeddings["apple"], embeddings["orange"])

print(f"Similarity (King, Queen):  {sim_king_queen:.4f}")
print(f"Similarity (King, Apple):  {sim_king_apple:.4f}")
print(f"Similarity (Apple, Orange): {sim_apple_orange:.4f}")

---

## 3. The Self-Attention Mechanism

Self-attention allows the model to scale word relationships dynamically using three matrices: Queries ($Q$), Keys ($K$), and Values ($V$):
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) \cdot V$$

- $Q \cdot K^T$ calculates similarity weights between all token pairs.
- Scaling by $\sqrt{d_k}$ stabilizes gradients during training.
- Softmax turns weights into probabilities (sum to 1).
- Multiplying by $V$ computes a weighted average representation.

In [ ]:
# Self-Attention Weight Computation Simulation
import numpy as np
from scipy.special import softmax

# 2 words in sequence, embedding dimension 4
# Sentence: "cat sat"
X = np.array([
    [1.0, 0.0, 2.0, 0.0],  # cat
    [0.0, 2.0, 0.0, 1.0]   # sat
])

# Mock Projection Matrices
W_Q = np.eye(4)
W_K = np.eye(4)
W_V = np.eye(4)

# Compute Q, K, V vectors
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Compute raw attention scores
scores = Q @ K.T

# Scale by sqrt(d_k) where d_k = 4
d_k = K.shape[1]
scaled_scores = scores / np.sqrt(d_k)

# Apply Softmax to get weights
weights = softmax(scaled_scores, axis=1)

# Compute Output
output = weights @ V

print("Attention Weights Matrix (Row: Query, Column: Key):")
print(weights)
print("\nOutput Representation (Context-Weighted Embeddings):")
print(output)

---

## 4. Building an In-Memory RAG Pipeline

Let's put everything together to build a complete **Retrieval-Augmented Generation (RAG)** system from scratch using Python.

### Documents Database
We have a small corpus of fact documents about space exploration.

In [ ]:
documents = [
    "The Apollo 11 mission landed humans on the Moon in July 1969. Neil Armstrong was the first person to walk on the lunar surface.",
    "Mars is the fourth planet from the Sun and is often called the Red Planet due to iron oxide on its surface.",
    "The James Webb Space Telescope (JWST) is a space telescope designed primarily to conduct infrared astronomy.",
    "Voyager 1 is a space probe launched by NASA in 1977 to study the outer Solar System and interstellar space."
]

# Simplified Vectorizer mapping keywords to index dimensions
vocab = ["apollo", "moon", "armstrong", "mars", "red", "planet", "telescope", "infrared", "voyager", "probe", "nasa"]

def vectorize(text):
    vec = np.zeros(len(vocab))
    for i, word in enumerate(vocab):
        if word in text.lower():
            vec[i] = 1.0
    return vec

# Create Document Embeddings
doc_embeddings = [vectorize(doc) for doc in documents]
print(f"Document 0 Embedding: {doc_embeddings[0]}")

### Query Retrieval & Grounded Prompt Assembly

In [ ]:
def retrieve_top_document(query):
    query_vec = vectorize(query)
    
    # Compute similarity with all docs
    similarities = []
    for doc_vec in doc_embeddings:
        if np.sum(doc_vec) == 0 or np.sum(query_vec) == 0:
            similarities.append(0.0)
        else:
            similarities.append(cosine_similarity(doc_vec, query_vec))
            
    best_idx = np.argmax(similarities)
    return documents[best_idx], similarities[best_idx]

query = "Tell me about the NASA voyager probe launched in 1977"
retrieved_doc, score = retrieve_top_document(query)

print(f"Query: '{query}'")
print(f"Retrieved Doc (Score {score:.2f}):\n  '{retrieved_doc}'\n")

# Construct RAG Grounding Prompt
prompt = f"""Answer the question using the provided context.

Context:
-----------
{retrieved_doc}
-----------

Question: {query}

Answer:
"""
print("Generated RAG Prompt:")
print(prompt)

### 🎓 Conceptual Review

**Question:** Why do we vector-compare a *query* to *document chunks* instead of the *entire document* at once?

*Double-click this cell to type your response here.*